# PCAP spot-check of CSE-CIC-IDS2018 DoS-SlowHTTPTest (Friday 16-02-2018)
Settings: Internet ON, no accelerator needed. Use **Save Version -> Save & Run All**. Every cell is fail-safe; if a later cell cannot run, the earlier output (zip member listing) is still useful - send it back.

In [ ]:
import subprocess, sys, os, re, json, time, shutil
def sh(c):
    print("$", c); r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-2000:]); return r
sh("pip install -q remotezip")
sh("DEBIAN_FRONTEND=noninteractive apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y tshark >/dev/null")
sh("tshark --version | head -1; capinfos --version | head -1")
OUT = "/kaggle/working/pcap_check"; os.makedirs(OUT, exist_ok=True)
TMP = "/kaggle/tmp" if os.path.isdir("/kaggle/tmp") else "/tmp"
print("scratch:", TMP, "free GB:", round(shutil.disk_usage(TMP).free / 1e9, 1))

In [ ]:
open('/kaggle/working/audit_degeneracy.py','w').write('#!/usr/bin/env python3\n"""Stand-alone vector-level audit of CIC flow CSV/Parquet files.\n\nReproduces Tables 4 and 5 of "Configuration Fingerprints in Flow-Based Slow-HTTP Detection":\ndistinct feature vectors per class and label conflicts under four schemas\n(30 canonical fields; all flow features; + destination port and protocol; + timestamp),\nplus SHA-256 digests of every input and the destination-port profile of each slow-HTTP class.\n\nUsage\n  python audit_degeneracy.py FILE [FILE ...] --out audit_out\n\nInputs (download first; no credentials needed):\n  aws s3 cp --no-sign-request "s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/Friday-16-02-2018_TrafficForML_CICFlowMeter.csv" .\n  aws s3 cp --no-sign-request "s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv" .\n  CICIDS2017 MachineLearningCVE: Wednesday-workingHours.pcap_ISCX.csv\n  CIC-DoS2017 (Parquet conversion of the CICFlowMeter output)\nRequires pandas (and pyarrow for Parquet).\n"""\nimport argparse, hashlib, json, os, sys\nimport pandas as pd\n\nRAW_FIELDS = {\n    "duration": ["Flow Duration"], "fwd_pkts": ["Total Fwd Packets", "Tot Fwd Pkts", "Total Fwd Packet"],\n    "bwd_pkts": ["Total Backward Packets", "Tot Bwd Pkts", "Total Bwd packets"],\n    "fwd_bytes": ["Total Length of Fwd Packets", "TotLen Fwd Pkts", "Fwd Packets Length Total", "Total Length of Fwd Packet"],\n    "bwd_bytes": ["Total Length of Bwd Packets", "TotLen Bwd Pkts", "Bwd Packets Length Total"],\n    "fwd_len_mean": ["Fwd Packet Length Mean", "Fwd Pkt Len Mean"], "fwd_len_max": ["Fwd Packet Length Max", "Fwd Pkt Len Max"],\n    "fwd_len_min": ["Fwd Packet Length Min", "Fwd Pkt Len Min"], "fwd_len_std": ["Fwd Packet Length Std", "Fwd Pkt Len Std"],\n    "bwd_len_mean": ["Bwd Packet Length Mean", "Bwd Pkt Len Mean"], "flow_byts_s": ["Flow Bytes/s", "Flow Byts/s"],\n    "flow_pkts_s": ["Flow Packets/s", "Flow Pkts/s"], "flow_iat_mean": ["Flow IAT Mean"], "flow_iat_std": ["Flow IAT Std"],\n    "flow_iat_max": ["Flow IAT Max"], "flow_iat_min": ["Flow IAT Min"], "fwd_iat_tot": ["Fwd IAT Total", "Fwd IAT Tot"],\n    "fwd_iat_mean": ["Fwd IAT Mean"], "fwd_iat_std": ["Fwd IAT Std"], "fwd_iat_max": ["Fwd IAT Max"], "fwd_iat_min": ["Fwd IAT Min"],\n    "bwd_iat_tot": ["Bwd IAT Total", "Bwd IAT Tot"], "bwd_iat_mean": ["Bwd IAT Mean"], "bwd_iat_std": ["Bwd IAT Std"],\n    "bwd_iat_max": ["Bwd IAT Max"], "bwd_iat_min": ["Bwd IAT Min"], "active_mean": ["Active Mean"], "active_std": ["Active Std"],\n    "idle_mean": ["Idle Mean"], "idle_std": ["Idle Std"]}\nMETA = {"timestamp", "dstport", "destinationport", "protocol", "flowid", "srcip", "sourceip", "dstip",\n        "destinationip", "srcport", "sourceport"}\nnorm = lambda s: "".join(ch for ch in str(s).lower() if ch.isalnum())\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for b in iter(lambda: f.read(1 << 20), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef load(path):\n    df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path, dtype=str, low_memory=False)\n    df = df.astype(str)\n    df.columns = [c.strip() for c in df.columns]\n    lab = [c for c in df.columns if norm(c) == "label"][0]\n    df = df[df[lab].map(norm) != "label"]          # repeated header rows\n    return df, lab\n\n\ndef canon_cols(cols):\n    lut = {norm(c): c for c in cols}\n    out = []\n    for k, cands in RAW_FIELDS.items():\n        hit = next((lut[norm(c)] for c in cands if norm(c) in lut), None)\n        if hit is None:\n            raise SystemExit("unresolved canonical field: " + k)\n        out.append(hit)\n    return out\n\n\ndef audit(path):\n    df, lab = load(path)\n    allc = [c for c in df.columns if c != lab]\n    meta = [c for c in allc if norm(c) in META]\n    flow = [c for c in allc if c not in meta]\n    pp = [c for c in meta if norm(c) in ("dstport", "destinationport", "protocol")]\n    schemas = {"30 canonical fields": canon_cols(allc), "all flow features": flow,\n               "flow + port/protocol": flow + pp, "all columns incl. timestamp": allc}\n    per_class, conflicts = [], []\n    for sname, cols in schemas.items():\n        h = pd.util.hash_pandas_object(df[cols], index=False).to_numpy()\n        d = pd.DataFrame({"h": h, "lab": df[lab].to_numpy()})\n        for L, g in d.groupby("lab"):\n            vc = g["h"].value_counts()\n            per_class.append({"file": os.path.basename(path), "schema": sname, "label": L, "rows": len(g),\n                              "distinct": len(vc), "mean_copies": round(len(g) / len(vc), 4),\n                              "top1_share": round(vc.iloc[0] / len(g), 4), "top2_share": round(vc.iloc[:2].sum() / len(g), 4)})\n        nl = d.groupby("h")["lab"].nunique()\n        bad = nl[nl > 1].index\n        pairs = d[d.h.isin(bad)].groupby("h")["lab"].agg(lambda s: " | ".join(sorted(set(s)))).value_counts()\n        conflicts.append({"file": os.path.basename(path), "schema": sname, "conflicting_vectors": int(len(bad)),\n                          "pairs": json.dumps(pairs.to_dict())})\n    ports = []\n    pc = next((c for c in df.columns if norm(c) in ("dstport", "destinationport")), None)\n    if pc:\n        for L, g in df[df[lab].str.contains("slow", case=False)].groupby(lab):\n            vc = g[pc].value_counts()\n            for port, n in vc.head(10).items():\n                ports.append({"file": os.path.basename(path), "label": L, "dst_port": port, "rows": int(n),\n                              "share": round(n / len(g), 4)})\n    return per_class, conflicts, ports, {"file": os.path.basename(path), "sha256": sha256(path),\n                                         "rows_after_header_removal": int(len(df)), "columns": len(df.columns) }\n\n\ndef main():\n    ap = argparse.ArgumentParser(); ap.add_argument("files", nargs="+"); ap.add_argument("--out", default="audit_out")\n    a = ap.parse_args(); os.makedirs(a.out, exist_ok=True)\n    P, C, D, H = [], [], [], []\n    for f in a.files:\n        print("auditing", f, flush=True)\n        p, c, d, h = audit(f); P += p; C += c; D += d; H.append(h)\n    pd.DataFrame(P).to_csv(os.path.join(a.out, "distinct_vectors_per_class.csv"), index=False)\n    pd.DataFrame(C).to_csv(os.path.join(a.out, "label_conflicts_by_schema.csv"), index=False)\n    pd.DataFrame(D).to_csv(os.path.join(a.out, "slow_class_dst_ports.csv"), index=False)\n    pd.DataFrame(H).to_csv(os.path.join(a.out, "input_checksums.csv"), index=False)\n    print(pd.DataFrame(P).query("label.str.contains(\'low\', case=False)", engine="python").to_string(index=False))\n    print(pd.DataFrame(C)[["file", "schema", "conflicting_vectors"]].to_string(index=False))\n    print(pd.DataFrame(D).to_string(index=False))\n    print(pd.DataFrame(H).to_string(index=False))\n\n\nif __name__ == "__main__":\n    main()\n')


In [ ]:
import traceback
try:
    import urllib.request
    CSV_URL = ("https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/"
               "Friday-16-02-2018_TrafficForML_CICFlowMeter.csv")
    dst = os.path.join(TMP, "Friday-16-02-2018_TrafficForML_CICFlowMeter.csv")
    if not os.path.exists(dst):
        urllib.request.urlretrieve(CSV_URL, dst)
    sh("python /kaggle/working/audit_degeneracy.py '{}' --out {}".format(dst, os.path.join(OUT, "csv_audit")))
except Exception:
    print("!! CSV audit FAILED"); traceback.print_exc()

In [ ]:
import traceback
try:
    from remotezip import RemoteZip
    BASE = "https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/"
    URL = BASE + "Original%20Network%20Traffic%20and%20Log%20data/Friday-16-02-2018/pcap.zip"
    with RemoteZip(URL) as z:
        INFO = [(i.filename, i.file_size, i.compress_size) for i in z.infolist()]
    print("members:", len(INFO), "| total uncompressed GB:", round(sum(s for _, s, _ in INFO) / 1e9, 1))
    for n, s, c in sorted(INFO, key=lambda t: -t[1])[:60]:
        print("  {:>9.2f} GB  {}".format(s / 1e9, n))
    json.dump(INFO, open(os.path.join(OUT, "pcap_zip_members.json"), "w"), indent=1)
except Exception:
    print("!! listing FAILED"); traceback.print_exc()

In [ ]:
import traceback
try:
    # The documented DoS victim is 18.217.21.148 (internal 172.31.69.25). Prefer members naming that address;
    # otherwise fall back to the largest member under the disk budget. The choice is printed.
    free = shutil.disk_usage(TMP).free
    budget = 0.45 * free
    cand = [t for t in INFO if "172.31.69.25" in t[0] and t[1] > 0 and t[1] < budget]
    if not cand:
        print("no member names 172.31.69.25 under budget; falling back to largest member under budget")
        cand = sorted([t for t in INFO if 0 < t[1] < budget and not t[0].endswith("/")], key=lambda t: -t[1])[:1]
    CHOSEN = cand
    print("chosen:", [(n, round(s / 1e9, 2)) for n, s, _ in CHOSEN])
    PCAPS = []
    with RemoteZip(URL) as z:
        for n, s, _ in CHOSEN:
            t0 = time.time(); p = z.extract(n, path=TMP); PCAPS.append(p)
            print("  extracted {} ({:.1f} GB) in {:.0f} s".format(n, s / 1e9, time.time() - t0))
except Exception:
    print("!! extraction FAILED"); traceback.print_exc()

In [ ]:
import traceback
try:
    import pandas as pd
    parts = []
    for p in PCAPS:
        txt = os.path.join(TMP, os.path.basename(p) + ".tsv")
        cmd = ('tshark -r "{}" -Y "tcp.port==21 || tcp.port==80" -T fields -E separator=/t '
               '-e tcp.stream -e frame.time_epoch -e ip.src -e ip.dst -e tcp.srcport -e tcp.dstport -e tcp.len > "{}"').format(p, txt)
        t0 = time.time(); sh(cmd); print("  tshark done in {:.0f} s, {:.2f} GB".format(time.time() - t0, os.path.getsize(txt) / 1e9))
        agg = []
        cols = ["stream", "t", "src", "dst", "sport", "dport", "plen"]
        for ch in pd.read_csv(txt, sep="\t", names=cols, chunksize=2_000_000, low_memory=False):
            ch = ch.dropna(subset=["stream", "t"])
            ch["plen"] = pd.to_numeric(ch["plen"], errors="coerce").fillna(0)
            g = ch.groupby("stream").agg(t0=("t", "min"), t1=("t", "max"), pkts=("t", "size"), payload=("plen", "sum"),
                                         src=("src", "first"), dst=("dst", "first"), sport=("sport", "first"), dport=("dport", "first"))
            agg.append(g)
        A = pd.concat(agg).groupby(level=0).agg(t0=("t0", "min"), t1=("t1", "max"), pkts=("pkts", "sum"), payload=("payload", "sum"),
                                                src=("src", "first"), dst=("dst", "first"), sport=("sport", "first"), dport=("dport", "first"))
        A["pcap"] = os.path.basename(p); parts.append(A.reset_index())
        os.remove(txt)
    C = pd.concat(parts, ignore_index=True)
    C["sport"] = pd.to_numeric(C.sport, errors="coerce"); C["dport"] = pd.to_numeric(C.dport, errors="coerce")
    C["server_port"] = C[["sport", "dport"]].min(axis=1)
    C["duration_s"] = C.t1 - C.t0
    C["t_utc"] = pd.to_datetime(C.t0, unit="s", utc=True)
    C["bin10"] = C.t_utc.dt.floor("10min")
    C["micro"] = (C.pkts <= 4) & (C.payload == 0) & (C.duration_s < 0.001)
    C["long"] = C.duration_s > 10
    C.to_csv(os.path.join(OUT, "tcp_streams_port21_80.csv.gz"), index=False)
    S = C.groupby(["bin10", "server_port"]).agg(streams=("pkts", "size"), micro_zero_payload=("micro", "sum"),
                                                 long_gt10s=("long", "sum"), median_dur_s=("duration_s", "median"))
    S.to_csv(os.path.join(OUT, "summary_by_10min_utc.csv"))
    pd.set_option("display.width", 220); print(S.to_string())
    for port in (21, 80):
        sub = C[C.server_port == port]
        print("\nport {}: top client -> server pairs".format(port))
        print(sub.groupby(["src", "dst"]).agg(n=("pkts", "size"), micro=("micro", "sum"), long=("long", "sum"))
              .sort_values("n", ascending=False).head(15).to_string())
except Exception:
    print("!! analysis FAILED"); traceback.print_exc()

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/pcap_check_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT): z.write(os.path.join(OUT, f), f)
print("download /kaggle/working/pcap_check_outputs.zip")